# 🚀 ETL Pipeline — Retenção de Clientes com IA

Pipeline ETL em Python que usa a **API do Claude (Anthropic)** para gerar mensagens de retenção personalizadas para cada cliente com base em seu perfil de uso.

---

## 🔄 Fluxo do Pipeline

```
sdw2023.csv
    │
    ▼
[ EXTRACT ]   → Lê o CSV e carrega os dados dos clientes
    │
    ▼
[ TRANSFORM ] → A API do Claude gera mensagens personalizadas
    │
    ▼
[ LOAD ]      → Salva o resultado em transformed_data.csv
```

---

## 🧠 Lógica de Personalização

| UsageScore | Perfil    | Estratégia                  |
|------------|-----------|--------------------------|
| < 20       | Em risco  | Oferecer benefício especial |
| 20 – 50    | Moderado  | Incentivar engajamento      |
| > 50       | Ativo     | Reconhecer e fidelizar      |

## ⚙️ Instalação das Dependências

In [ ]:
!pip install pandas anthropic -q

## 🔑 Configuração da API Key

> ⚠️ **Nunca** coloque sua chave diretamente no código.  
> Crie sua chave em: https://console.anthropic.com/

In [ ]:
import os
import anthropic
import pandas as pd
from datetime import datetime

# Configure sua chave como variável de ambiente antes de rodar:
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("✅ Cliente da API configurado com sucesso!")

---

## 1️⃣ EXTRACT — Extração dos Dados

Lemos o arquivo `sdw2023.csv` e carregamos os dados dos clientes em memória.

In [ ]:
def extract_users(file_path: str) -> list[dict]:
    """Lê o CSV e retorna uma lista de dicionários com os dados dos usuários."""
    df = pd.read_csv(file_path)
    print(f"✅ {len(df)} usuários carregados com sucesso!")
    return df

df_users = extract_users("sdw2023.csv")

# Visualizando os dados extraídos
df_users

### 📊 Explorando os Dados

In [ ]:
print("📋 Informações do Dataset:")
print(f"  Total de clientes: {len(df_users)}")
print(f"  Colunas: {list(df_users.columns)}")
print()
print("📊 Distribuição por Plano:")
print(df_users['Plan'].value_counts().to_string())
print()
print("📈 Estatísticas do UsageScore:")
print(df_users['UsageScore'].describe().to_string())

In [ ]:
# Classificando os clientes por perfil de risco
def classify_user(score):
    if score < 20:
        return "🔴 Em risco"
    elif score < 50:
        return "🟡 Moderado"
    else:
        return "🟢 Ativo"

df_users['Perfil'] = df_users['UsageScore'].apply(classify_user)
print("👥 Distribuição por Perfil de Risco:")
print(df_users['Perfil'].value_counts().to_string())
df_users[['Name', 'Plan', 'UsageScore', 'Perfil']]

---

## 2️⃣ TRANSFORM — Geração de Mensagens com IA

Para cada cliente, enviamos o perfil para a **API do Claude** que gera uma mensagem de retenção personalizada e humanizada.

In [ ]:
def generate_message_with_claude(user: dict) -> str:
    """Usa a API do Claude para gerar uma mensagem de retenção personalizada."""
    prompt = f"""
Você é um especialista em Customer Success de uma empresa de SaaS.
Gere uma mensagem de retenção curta, amigável e personalizada em português
para o seguinte cliente:

- Nome: {user['Name']}
- Plano: {user['Plan']}
- Score de uso (0-100): {user['UsageScore']}
- Meses como cliente: {user['MonthsActive']}
- Último login: {user['LastLogin']}

Regras:
- Score abaixo de 20: cliente em risco, ofereça um benefício especial.
- Score entre 20 e 50: cliente moderado, incentive o engajamento.
- Score acima de 50: cliente ativo, reconheça e fidelize.
- Máximo de 3 frases. Não use emojis em excesso. Seja direto e humano.
"""
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text.strip()


def transform_users(df: pd.DataFrame) -> pd.DataFrame:
    """Itera sobre os usuários e adiciona a mensagem gerada pelo Claude."""
    users = df.to_dict(orient="records")
    print(f"🤖 Gerando mensagens personalizadas com Claude para {len(users)} clientes...\n")
    
    for i, user in enumerate(users, start=1):
        try:
            user["Message"] = generate_message_with_claude(user)
            print(f"  [{i}/{len(users)}] ✅ {user['Name']} ({user['Perfil']})")
        except Exception as e:
            user["Message"] = "Erro ao gerar mensagem."
            print(f"  [{i}/{len(users)}] ❌ Erro para {user['Name']}: {e}")
    
    print("\n✅ Transformação concluída!")
    return pd.DataFrame(users)


df_transformed = transform_users(df_users)

In [ ]:
# Visualizando as mensagens geradas
print("💬 Mensagens geradas pelo Claude:\n")
for _, row in df_transformed.iterrows():
    print(f"👤 {row['Name']} | {row['Perfil']}")
    print(f"   {row['Message']}")
    print()

---

## 3️⃣ LOAD — Salvando os Resultados

Salvamos o DataFrame transformado em um novo arquivo CSV com todas as mensagens geradas.

In [ ]:
def save_data(df: pd.DataFrame, output_path: str) -> None:
    """Salva os dados transformados em um novo arquivo CSV."""
    df.to_csv(output_path, index=False)
    print(f"✅ Dados salvos em '{output_path}' com sucesso!")

save_data(df_transformed, "transformed_data.csv")

# Visualizando o resultado final
print(f"\n📁 Resultado final — {len(df_transformed)} clientes processados:")
df_transformed[['Name', 'Plan', 'UsageScore', 'Perfil', 'Message']]

---

## ✅ Pipeline Concluído!

O arquivo `transformed_data.csv` foi gerado com as mensagens personalizadas para cada cliente.

### 🛠️ Tecnologias utilizadas
- **Python 3.10+**
- **Pandas** — manipulação de dados
- **Anthropic SDK** — integração com a API do Claude

### 💡 Melhorias futuras
- [ ] Adicionar visualizações com Matplotlib
- [ ] Envio real de e-mails com SMTP
- [ ] Dashboard interativo com Streamlit
- [ ] Testes unitários com Pytest